In [3]:
import sys
from pathlib import Path
import pandas as pd

def get_project_root(path):
    """
    Get the root directory of the project.
    """
    project_root = Path.cwd().resolve()
    if not (project_root / path).exists() and (project_root.parent / path).exists():
        project_root = project_root.parent

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    return project_root

 

get_project_root("src")
get_project_root("data")


WindowsPath('C:/Users/gabri/Documents/capstone-projects/airline-support-bot/backend/rag-service')

In [4]:
from src.rag.embedding_models.embedding_factory import EmbeddingFactory
from src.rag.vector_store.instance import vector_store
from src.rag.rerankers.instance import reranker
from src.rag.llms.llama import LlamaManager
from src.rag.evaluation_metrics.semantic_evaluator import SemanticRelevanceEvaluator
from src.rag.evaluation_metrics.evaluation import calculate_precision,calculate_hit_at_k,calculate_mrr
from data.raw.questions.question_dataset import QUESTION_DATASET
from data.raw.questions.questiondataset import questions 



In [5]:
embedding_manager = EmbeddingFactory.create_embedding_model("gemini") 


In [ ]:

#  Create embeddings for ALL questions first

question_texts = [
    question["question_text"]
    for question in questions
]

query_embeddings = embedding_manager.create_query_embedding(
    question_texts
)
# 2. Evaluation

evaluation_results = []

semantic_evaluator = SemanticRelevanceEvaluator()


for idx, question in enumerate(questions):

    query = question["question_text"]

    relevant_text = question["relevant_text"]


    
    # 1. Get the PRE-COMPUTED query embedding
    query_embedding = query_embeddings[idx]


    # --------------------------------------------------------
    # 2. Retrieve top K
    # --------------------------------------------------------

    results = vector_store.retrieve(
        query_embedding,
        top_k=5
    )


    # --------------------------------------------------------
    # 3. Build ranked retrieved chunks
    # --------------------------------------------------------

    retrieved_chunks = []

    for i in range(len(results["documents"][0])):

        chunk = {
            "rank": i + 1,

            "chunk_id": results["ids"][0][i],

            "section": (
                results["metadatas"][0][i]
                .get("section")
            ),

            "distance": (
                results["distances"][0][i]
            ),

            "content": (
                results["documents"][0][i]
            )
        }

        retrieved_chunks.append(chunk)


     
    #  Semantic relevance evaluation
    # Each retrieved chunk is compared against relevant_text
    # using your SemanticRelevanceEvaluator.
    

    retrieved_chunks = semantic_evaluator.evaluate(
        query=query,
        retrieved_chunks=retrieved_chunks
    )

 
    # 5. Extract relevance labels


    relevance_labels = [
        chunk["is_relevant"]
        for chunk in retrieved_chunks
    ]


    # --------------------------------------------------------
    # 6. Calculate metrics
    # --------------------------------------------------------

    precision = calculate_precision(
        relevance_labels
    )

    hit = calculate_hit_at_k(
        relevance_labels
    )

    reciprocal_rank = calculate_mrr(
        relevance_labels
    )


    # --------------------------------------------------------
    # 7. Store detailed evaluation result
    # --------------------------------------------------------

    evaluation_results.append({

        "question_id": question["question_id"],

        "question": query,

        "relevant_text": relevant_text,

        "source_document": (
            question["source_document"]
        ),

        "retrieved_chunks": retrieved_chunks,

        "precision": precision,

        "hit": hit,

        "reciprocal_rank": reciprocal_rank
    })

In [ ]:
test_questions = questions[600:608]

question_texts = [
    question["question_text"]
    for question in test_questions
]

query_embeddings = embedding_manager.create_query_embedding(
    question_texts
)

evaluation_results = []

semantic_evaluator = SemanticRelevanceEvaluator()

for idx, question in enumerate(test_questions):

    query = question["question_text"]
    relevant_text = question["relevant_text"]

    query_embedding = query_embeddings[idx]

    results = vector_store.retrieve(
        query_embedding,
        top_k=5
    )

    retrieved_chunks = []

    for i in range(len(results["documents"][0])):

        chunk = {
            "rank": i + 1,
            "chunk_id": results["ids"][0][i],
            "section": results["metadatas"][0][i].get("section"),
            "distance": results["distances"][0][i],
            "content": results["documents"][0][i]
        }

        retrieved_chunks.append(chunk)

    retrieved_chunks = semantic_evaluator.evaluate(
        query=query,
        retrieved_chunks=retrieved_chunks
    )

    relevance_labels = [
        chunk["is_relevant"]
        for chunk in retrieved_chunks
    ]

    precision = calculate_precision(relevance_labels)
    hit = calculate_hit_at_k(relevance_labels)
    reciprocal_rank = calculate_mrr(relevance_labels)

    evaluation_results.append({
        "question_id": question["question_id"],
        "question": query,
        "relevant_text": relevant_text,
        "source_document": question["source_document"],
        "retrieved_chunks": retrieved_chunks,
        "precision": precision,
        "hit": hit,
        "reciprocal_rank": reciprocal_rank
    })

In [ ]:
def inspect_question(question_id):

    result = next(
        result
        for result in evaluation_results
        if result["question_id"] == question_id
    )

    print("=" * 80)
    print("QUESTION")
    print("=" * 80)
    print(result["question"])

    print("\nEXPECTED SECTION")
    print("=" * 80)
    print(result["source_document"])

    print("\nRETRIEVED CHUNKS")
    print("=" * 80)

    for chunk in result["retrieved_chunks"]:

        print(f"\nRank: {chunk['rank']}")
        print(f"Chunk ID: {chunk['chunk_id']}")
        print(f"Section: {chunk['section']}")
        print(f"Distance: {chunk['distance']}")
        print(f"Relevant: {chunk['is_relevant']}")

        print("\nContent:")
        print(chunk["content"])

        print("-" * 80)

In [ ]:
inspect_question("Q601")

In [ ]:
retrieved_chunks = []
question_text = questions[11]["question_text"]
query_embedding = embedding_manager.create_query_embedding(
    [question_text]
)[0]
results = vector_store.retrieve(
    query_embedding,
    top_k=5
)
for i in range(len(results["documents"][0])):

        chunk = {
            "rank": i + 1,

            "chunk_id": results["ids"][0][i],

            "section": (
                results["metadatas"][0][i]
                .get("section")
            ),

            "distance": (
                results["distances"][0][i]
            ),

            "content": (
                results["documents"][0][i]
            )
        }

        retrieved_chunks.append(chunk)

reranked_chunks = reranker.rerank(question_text,retrieved_chunks)
chunk = [chunk for chunk in reranked_chunks if chunk["reranked_rank"] == 1]
content = chunk[0].get("content")

print (f"Question: {question_text}")
for chunk in reranked_chunks:
    print(f"Rank: {chunk['rank']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Section: {chunk['section']}")
    print(f"Distance: {chunk['distance']}")
    print(f"Content: {chunk['content']}")
    print("-" * 80)
      


Processing queries 1 to 1
Embeddings returned: 1
Question: What departure and destination information should the agent confirm?
Rank: 1
Chunk ID: KQ-BOOK-006-chunk-065
Section: Customer Service Procedure for Flight Booking Queries
Distance: 0.486820250749588
Content: Context: This TARGET CHUNK outlines the step-by-step process for a customer service agent to assist a passenger with a flight booking query, providing a structured approach to resolving queries. It fits within the surrounding document context as a procedural guide for customer service agents, following the introduction of the document's purpose and scope.

1. Confirm the passenger's preferred departure location and destination.
2. Confirm the intended travel dates.
3. Confirm the trip type: one-way, return, or multi-city.
4. Confirm the passenger's preferred travel class.
5. Confirm the number of passengers.
6. Check available flights and fares using the approved booking system.
7. Explain the available options and applica

In [27]:
llm = LlamaManager()
response =llm.generate_response(query=question_text,context=content)
print(question_text)
print("Response:")
print(response)

What factors can affect the changes I make to an existing booking?
Response:
According to our Customer Service Procedure, the following factors can affect the changes you make to an existing booking:

* Fare conditions: The requested change must be permitted under the fare conditions.
* Availability of alternative flights: If a date or flight change is requested, you must check available alternative flights.
* Fare difference or change charges: You must inform the passenger of any applicable fare difference or change charges.

These factors will impact the changes you can make to an existing booking and the process you need to follow to ensure a smooth and successful modification or cancellation.
